# ZQE with Poisson MAP Encoder + L1 penalty

Reproduces `zqe_gaussian_map_sparse.ipynb` but uses:
- **`MapEncoderPoissonNewton`** — exact Poisson MAP z̃ (Newton GLM solver)
- **`ZQEAutoFitter`** — standard API (warm-up Adam → SGD + Ruppert–Polyak)
- **`l2 = c/n` ridge stabilizer** — prevents near-unidentified divergence
- **`l1` proximal operator** — ISTA soft-threshold after every SGD step

Setting: **overcomplete** fitted model (q_fit > q_true) with a sparse true W.
Goal: L1 drives extra columns toward zero, recovering the true sparsity pattern.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy
import numpy as np
import torch
import matplotlib.pyplot as plt

from gllvm.simulations import make_sparse, simulate
from gllvm.autofit import ZQEAutoFitter, procrustes_error
from gllvm.encoder import MapEncoderPoissonNewton

SEED   = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- experiment config ---
Q_TRUE = 3      # true latent dim
Q_FIT  = 6      # fitted (overcomplete) — extra columns should shrink to 0
P      = 200    # responses
N      = 500    # observations
WZ_SCALE = 0.5
RPL    = P // 2  # responses per latent (50% sparse true W)

# ZQE hyperparams
L2     = 1e-3 / N   # c/n ridge stabilizer
L1     = 0.002      # proximal L1 threshold multiplier

print(f"device={device}  Q_TRUE={Q_TRUE}  Q_FIT={Q_FIT}  P={P}  N={N}")
print(f"l2={L2:.2e}  l1={L1:.3g}")

## Data

In [ ]:
torch.manual_seed(SEED)
g_true = make_sparse(
    n_latent=Q_TRUE, poisson=P, active_latent=Q_TRUE,
    wz_scale=WZ_SCALE, responses_per_latent=RPL,
).to(device)

Y, _ = simulate(g_true, n_samples=N, device=device)
W_true = g_true.wz.detach().cpu()

prop_zero = float((W_true == 0).float().mean())
print(f"Y: {Y.shape}   mean count: {Y.float().mean():.2f}")
print(f"W_true: {W_true.shape}   prop zero: {prop_zero:.1%}")

## Fit — overcomplete ZQE with L1

In [ ]:
from gllvm.simulations import make_sparse
from gllvm.gllvm_module import GLLVM
from gllvm.glms import PoissonGLM

def fresh_model(q, p, wz_scale, seed=0):
    torch.manual_seed(seed)
    g = GLLVM(latent_dim=q, output_dim=p)
    g.add_glm(PoissonGLM, idx=range(p), params={"T": torch.log1p})
    torch.nn.init.normal_(g.wz, std=wz_scale)
    torch.nn.init.zeros_(g.bias)
    return g.to(device)


def run(q_fit, l1, l2, label, l1_jitter=0.1):
    g = fresh_model(q_fit, P, WZ_SCALE)
    ft = ZQEAutoFitter(
        g,
        encoder_factory=lambda g: MapEncoderPoissonNewton(g, lam=1.0, max_iter=30),
        device=device,
        l2=l2,
        l1=l1,
        l1_jitter=l1_jitter,  # revive dead columns at each restart
        steps_per_round=150,
        max_rounds=4,
        refine_lr=0.5,
        ema_decay=0.95,
        verbose=True,
        store_wz_trace=False,
        seed=SEED,
    ).fit(Y)
    W_hat = ft.model.wz.detach().cpu()
    zeros = float((W_hat.abs() < 1e-8).float().mean())
    print(f"[{label}]  zeros in Ŵ: {zeros:.1%}  (true: {prop_zero:.1%})")
    return ft, W_hat


ft_l1,    W_l1    = run(Q_FIT, l1=L1,  l2=L2,  label="overcomplete + L1")
ft_dense, W_dense = run(Q_FIT, l1=0.0, l2=L2,  label="overcomplete, no L1")

## W heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
vmax = W_true.abs().max().item()

for ax, W, title in zip(axes,
    [W_true, W_l1, W_dense],
    [f"W_true  (q={Q_TRUE})",
     f"Ŵ overcomplete + L1  (q={Q_FIT})",
     f"Ŵ overcomplete, no L1  (q={Q_FIT})"]):
    im = ax.imshow(W.numpy(), aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Latent dim"); ax.set_ylabel("Response")

fig.colorbar(im, ax=axes[-1], shrink=0.8)
fig.suptitle(f"Overcomplete ZQE (Poisson-MAP encoder)  P={P} N={N}", fontsize=11)
plt.tight_layout()
plt.show()

## Sparsity recovery

In [ ]:
from gllvm.autofit import orthogonal_align

W_true_np = W_true.numpy()

for W_hat, label in [(W_l1, "L1"), (W_dense, "no L1")]:
    W_np = W_hat.numpy()
    zeros = float((np.abs(W_np) < 1e-8).mean())
    print(f"\n[{label}]  exact zeros: {zeros:.1%}")

    # percentile thresholds as sparsity estimates
    for pct in [50, 70, 80]:
        thresh = np.percentile(np.abs(W_np), pct)
        est_nz = (np.abs(W_np) > thresh).astype(int)
        # compare against broadcast of true W (true has Q_TRUE cols, fitted has Q_FIT)
        # just report what fraction of entries are thresholded to zero
        print(f"  p{pct} thresh={thresh:.3f}  estimated zeros: {1 - est_nz.mean():.1%}")

## Column norms — do extra columns collapse?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

for ax, W_hat, label in [(ax1, W_l1, "L1"), (ax2, W_dense, "no L1")]:
    col_norms = W_hat.norm(dim=0).numpy()
    ax.bar(range(Q_FIT), col_norms)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"Column norms of Ŵ — {label}")
    ax.set_xlabel("Fitted latent dim")
    ax.set_ylabel("||ŵ_k||")

fig.suptitle(f"Overcomplete q={Q_FIT}, true q={Q_TRUE} — do extra columns collapse?")
plt.tight_layout()
plt.show()